# Dataset loader: IPG and PPG waveforms with reference blood pressure

This notebook loads the dataset described in the README of this folder from the Google Cloud Storage bucket `gs://ipg-obp-dataset-release`.

You need an account that has been approved for access. See "Data availability" in the README.

**Before you share this notebook, clear its outputs.** The outputs contain participant data.

## 1. Sign in and connect

In [ ]:
import io
import os
import re

from google.cloud import storage
import matplotlib.pyplot as plt
import pandas as pd

# In Google Colab, sign in with the account that was approved for access.
# Outside Colab, run `gcloud auth application-default login` first.
if 'COLAB_RELEASE_TAG' in os.environ:
  from google.colab import auth
  auth.authenticate_user()

# Set this to a Google Cloud project that you can use.
PROJECT_ID = 'your-gcp-project-id'  # @param {type: "string"}
BUCKET_NAME = 'ipg-obp-dataset-release'
SAMPLE_RATE_HZ = 2000

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

## 2. Participant tables

`demographic_data_release_v1.csv` has one row per participant. `train_test_split_mapping.csv` assigns each participant to `Train` or `Test`.

The BP status used in the manuscript is derived here: `HBP` is true if SBP is 130 mmHg or more, or DBP is 80 mmHg or more.

Expected: 261 participants; `Train` has 110 false and 111 true; `Test` has 21 false and 19 true.

In [ ]:
def read_csv(name):
  """Reads one CSV file from the bucket. PIN is kept as text."""
  data = bucket.blob(name).download_as_bytes()
  return pd.read_csv(io.BytesIO(data), dtype={'PIN': str})


df_demo = read_csv('demographic_data_release_v1.csv')
df_split = read_csv('train_test_split_mapping.csv')
df_meta = df_demo.merge(df_split, on='PIN', how='inner', validate='one_to_one')
df_meta['HBP'] = (df_meta['SBP (mmHg)'] >= 130) | (df_meta['DBP (mmHg)'] >= 80)

print(f'Participants: {len(df_meta)}')
print(pd.crosstab(df_meta['Split'], df_meta['HBP']))

## 3. File inventory

Each waveform file is named `biopac_parsed_parquet/<PIN>_collection<N>.parquet`, with N from 1 to 4.

Expected: 1042 files; 261, 261, 260 and 260 files for collections 1 to 4; two participant and collection pairs without a file.

In [ ]:
FILE_RE = re.compile(r'^biopac_parsed_parquet/(\d+)_collection([1-4])\.parquet$')

available = {}  # (PIN, collection number) -> file size in bytes
for blob in bucket.list_blobs(prefix='biopac_parsed_parquet/'):
  match = FILE_RE.match(blob.name)
  if match:
    available[(match.group(1), int(match.group(2)))] = blob.size

files_per_collection = pd.Series([c for _, c in available]).value_counts().sort_index()
missing = [
    (pin, c) for pin in df_meta['PIN'] for c in (1, 2, 3, 4) if (pin, c) not in available
]

print(f'Waveform files: {len(available)}')
print('Files per collection:')
print(files_per_collection.to_string())
print(f'Participant and collection pairs without a file: {missing}')
print(f'Total size: {sum(available.values()) / 1e9:.1f} GB')

## 4. Waveforms

All recordings are sampled at 2000 Hz. The channels in one file were recorded at the same time. Different collections are separate recordings, so their time axes are not aligned with each other.

The cell below loads every available collection for one participant. The last cell plots a 10 second window from collections 4 and 1.

In [ ]:
def load_waveform(pin, collection):
  """Returns one recording as a DataFrame. Collections are numbered 1 to 4."""
  if (pin, collection) not in available:
    raise FileNotFoundError(f'No collection {collection} file for participant {pin}.')
  blob = bucket.blob(f'biopac_parsed_parquet/{pin}_collection{collection}.parquet')
  return pd.read_parquet(io.BytesIO(blob.download_as_bytes()))


pin = df_meta['PIN'].iloc[0]
recordings = {c: load_waveform(pin, c) for c in (1, 2, 3, 4) if (pin, c) in available}

for c, df in recordings.items():
  print(
      f'Collection {c}: {len(df)} samples, {len(df) / SAMPLE_RATE_HZ:.1f} s, '
      f'columns {list(df.columns)}'
  )

In [ ]:
def window(df, start_s=20.0, end_s=30.0):
  """Returns the rows of a recording between two times, in seconds."""
  return df[(df['time'] >= start_s) & (df['time'] <= end_s)]


panels = [
    (4, 'ECG', 'Collection 4: ECG (mV)'),
    (4, 'IPG_integrated_dZdt_flattened', 'Collection 4: IPG (Ohm)'),
    (4, 'PPG_flattened', 'Collection 4: PPG (mV)'),
    (1, 'IPG_integrated_dZdt_flattened', 'Collection 1: IPG (Ohm)'),
    (1, 'PPG_flattened', 'Collection 1: PPG (mV)'),
]
panels = [p for p in panels if p[0] in recordings]

fig, axes = plt.subplots(
    len(panels), 1, figsize=(12, 2.2 * len(panels)), sharex=True, squeeze=False
)
for ax, (c, column, label) in zip(axes[:, 0], panels):
  w = window(recordings[c])
  ax.plot(w['time'], w[column], lw=1.0)
  ax.set_ylabel(label)
  ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel('Time since the start of each recording (s)')
plt.tight_layout()
plt.show()